# Probability Sampling Methods in Python – Complete Guide

**Goal:** Clear explanations + modern, recommended Python code for the most important probability sampling techniques.

### Sampling Methods Covered
1. Simple Random Sampling
2. Systematic Sampling
3. Stratified Sampling
4. Cluster Sampling
5. Multi-stage Sampling
6. Bonus: Reservoir Sampling & Weighted Random Sampling

> All examples use the modern `numpy.random.Generator` (`default_rng`) which is the recommended way since NumPy 1.17.


## Quick Overview

| Method                  | Core Idea                                      | Best When...                              | Main Risk / Limitation                  |
|-------------------------|------------------------------------------------|-------------------------------------------|-----------------------------------------|
| **Simple Random**       | Every unit has equal chance                    | Homogeneous population                    | May miss important subgroups            |
| **Systematic**          | Select every *k*-th item                       | Ordered list, want even coverage          | Periodicity / hidden patterns           |
| **Stratified**          | Sample from **every** important group          | Need representation of key groups         | Requires knowing the strata             |
| **Cluster**             | Randomly select whole groups, take everyone    | Geographically dispersed population       | Higher sampling error                   |
| **Multi-stage**         | Combine methods in successive stages           | Very large hierarchical populations       | Complex variance estimation             |


## 1. Simple Random Sampling (SRS)

**Definition**  
Every individual in the population has an **equal** and **independent** chance of being selected.

**When to use**
- Population is relatively homogeneous
- You have a complete list of the population
- You want the theoretically simplest unbiased method

**Advantages**
- Unbiased
- Easy to understand and implement
- Straightforward statistical inference

**Disadvantages**
- Can be expensive if population is spread out
- May under-represent important minority groups by chance


In [ ]:
import numpy as np
import pandas as pd

# Modern random generator (always preferred)
rng = np.random.default_rng(seed=42)

# -------------------------------------------------
# Create a simple population
# -------------------------------------------------
population = pd.DataFrame({
    'id': np.arange(1, 1001),
    'value': rng.normal(loc=100, scale=15, size=1000)
})

print("Population size:", len(population))

# -------------------------------------------------
# Simple Random Sample of size 100
# -------------------------------------------------
sample_size = 100

# Method 1: Using pandas (most common in data analysis)
srs_sample = population.sample(n=sample_size, random_state=42)

# Method 2: Pure NumPy style
indices = rng.choice(population.index, size=sample_size, replace=False)
srs_sample_np = population.loc[indices]

print("\nSample size:", len(srs_sample))
print(srs_sample.head(8))


## 2. Systematic Sampling

**Definition**  
Select every **k-th** element from an ordered list, after choosing a random starting point.

$$k = \left\lfloor \frac{N}{n} \right\rfloor$$

**When to use**
- You have an ordered list (customer list, street addresses, time series…)
- You want the sample to be evenly spread across the population
- Simplicity is important

**Main Danger**  
If the population has a **periodic pattern** that matches the interval $k$, the sample can become biased.


In [ ]:
# -------------------------------------------------
# Systematic Sampling
# -------------------------------------------------
rng = np.random.default_rng(seed=42)

population = pd.DataFrame({
    'id': np.arange(1, 1001),
    'score': rng.integers(50, 100, size=1000)
})

N = len(population)
n = 100
k = N // n                       # sampling interval

# Random starting point between 0 and k-1
start = rng.integers(0, k)

# Select every k-th element
systematic_sample = population.iloc[start::k].head(n)

print(f"Population size N = {N}")
print(f"Desired sample size n = {n}")
print(f"Sampling interval k = {k}")
print(f"Random start index = {start}")
print(f"\nSystematic sample size: {len(systematic_sample)}")
print(systematic_sample.head(10))


## 3. Stratified Sampling

**Definition**  
The population is divided into homogeneous groups called **strata**.  
You then draw a random sample **from every stratum** (usually proportionally).

**When to use**
- Important subgroups exist (gender, age group, region, customer type…)
- You want to guarantee representation of every group
- Subgroups have different means / variances

**Two common versions**
- **Proportional** allocation (most common)
- **Optimal** allocation (when variances differ a lot)


In [ ]:
# -------------------------------------------------
# Stratified Sampling
# -------------------------------------------------
rng = np.random.default_rng(seed=42)

# Create population with a clear stratum
df = pd.DataFrame({
    'student_id': range(1, 501),
    'gender': rng.choice(['Male', 'Female'], size=500, p=[0.45, 0.55]),
    'score': rng.normal(75, 12, 500).round(1)
})

print("Original gender distribution:")
print(df['gender'].value_counts(normalize=True).round(3))
print()

# ---------- Method A: scikit-learn (excellent) ----------
from sklearn.model_selection import train_test_split

stratified_sample, _ = train_test_split(
    df,
    train_size=0.2,               # keep 20%
    stratify=df['gender'],
    random_state=42
)

print("Stratified sample (sklearn) - gender distribution:")
print(stratified_sample['gender'].value_counts(normalize=True).round(3))
print()

# ---------- Method B: Pure pandas (also excellent) ----------
stratified_pandas = (
    df.groupby('gender', group_keys=False)
      .apply(lambda x: x.sample(frac=0.2, random_state=42))
)

print("Stratified sample (pandas) - gender distribution:")
print(stratified_pandas['gender'].value_counts(normalize=True).round(3))
print("\nSample size:", len(stratified_pandas))


## 4. Cluster Sampling

**Definition**  
1. Divide the population into clusters (usually natural groups).  
2. Randomly select **some clusters**.  
3. Take **all** individuals inside the selected clusters.

**When to use**
- Population is geographically dispersed
- It is expensive to travel to many locations
- You only have a list of clusters, not of individuals

**Important difference from Stratified Sampling**
- Stratified → sample **from every** group  
- Cluster → sample **some whole** groups


In [ ]:
# -------------------------------------------------
# Cluster Sampling
# -------------------------------------------------
rng = np.random.default_rng(seed=42)

# Population: 8 schools, 40 students each
schools = [f"School_{chr(65+i)}" for i in range(8)]  # School_A ... School_H
df = pd.DataFrame({
    'student_id': range(1, 321),
    'school': np.repeat(schools, 40),
    'score': rng.integers(55, 100, size=320)
})

print("Population size:", len(df))
print("Number of clusters (schools):", df['school'].nunique())
print()

# Step 1: List all clusters
all_clusters = df['school'].unique()

# Step 2: Randomly select 3 clusters
selected_clusters = rng.choice(all_clusters, size=3, replace=False)
print("Selected clusters:", selected_clusters)

# Step 3: Take ALL students from the selected clusters
cluster_sample = df[df['school'].isin(selected_clusters)].copy()

print(f"\nCluster sample size: {len(cluster_sample)}")
print(cluster_sample.groupby('school').size())
print("\nFirst rows of the sample:")
print(cluster_sample.head(8))


## 5. Multi-stage Sampling

**Definition**  
Sampling is performed in **several successive stages**.  
Usually a combination of cluster sampling + simple random (or systematic) sampling at lower levels.

**Classic example**
1. Select regions
2. Within selected regions → select cities
3. Within selected cities → select neighborhoods
4. Within selected neighborhoods → select households

**When to use**
- Very large populations
- Strong geographic hierarchy
- Cost and logistics are major constraints


In [ ]:
# -------------------------------------------------
# Multi-stage Sampling
# -------------------------------------------------
rng = np.random.default_rng(seed=42)

# Create hierarchical population
records = []
for region in range(1, 5):                 # 4 regions
    for city in range(1, 6):               # 5 cities per region
        for hh in range(1, 81):            # 80 households per city
            records.append({
                'region': f'Region_{region}',
                'city': f'City_{region}_{city}',
                'household_id': f'H{region}_{city}_{hh:03d}',
                'income': rng.normal(45000, 12000)
            })

population = pd.DataFrame(records)
print("Full population size:", len(population))
print("Regions:", population['region'].nunique())
print("Cities:", population['city'].nunique())
print()

# ----- Stage 1: Select 2 regions -----
selected_regions = rng.choice(
    population['region'].unique(),
    size=2,
    replace=False
)
print("Stage 1 – Selected regions:", selected_regions)

# ----- Stage 2: Select 2 cities from each selected region -----
selected_cities = []
for reg in selected_regions:
    cities = population.loc[population['region'] == reg, 'city'].unique()
    chosen = rng.choice(cities, size=2, replace=False)
    selected_cities.extend(chosen)

print("Stage 2 – Selected cities:", selected_cities)

# ----- Stage 3: Select 8 households from each selected city -----
samples = []
for city in selected_cities:
    city_data = population[population['city'] == city]
    samples.append(city_data.sample(n=8, random_state=rng))

multi_stage_sample = pd.concat(samples).reset_index(drop=True)

print(f"\nFinal multi-stage sample size: {len(multi_stage_sample)}")
print(multi_stage_sample.head(12))


## 6. Bonus Methods

### 6.1 Reservoir Sampling
Useful when the population is a **stream** (you don’t know the total size in advance) or is too large to fit in memory.

### 6.2 Weighted Random Sampling
Each unit has a different probability of being selected (useful for importance sampling or unequal probability designs).


In [ ]:
# -------------------------------------------------
# Reservoir Sampling (classic algorithm)
# -------------------------------------------------
def reservoir_sampling(stream, k, rng=None):
    """Return a simple random sample of size k from a stream."""
    if rng is None:
        rng = np.random.default_rng()
    
    reservoir = []
    for i, item in enumerate(stream):
        if i < k:
            reservoir.append(item)
        else:
            # Probability k / (i+1) of replacing an existing item
            j = rng.integers(0, i + 1)
            if j < k:
                reservoir[j] = item
    return reservoir

# Example
stream = range(10_000)          # imagine a huge stream
sample = reservoir_sampling(stream, k=10, rng=np.random.default_rng(42))
print("Reservoir sample of 10 items from a stream of 10,000:")
print(sample)


In [ ]:
# -------------------------------------------------
# Weighted Random Sampling
# -------------------------------------------------
rng = np.random.default_rng(42)

items = np.array(['A', 'B', 'C', 'D', 'E'])
weights = np.array([0.1, 0.1, 0.2, 0.3, 0.3])   # higher weight → higher chance

# Sample with replacement according to weights
weighted_sample = rng.choice(items, size=15, replace=True, p=weights)
print("Weighted sample:")
print(weighted_sample)
print("\nFrequency:")
print(pd.Series(weighted_sample).value_counts(normalize=True).round(3))


## Decision Guide – Which Sampling Method Should I Use?

| Situation                                              | Recommended Method          |
|--------------------------------------------------------|-----------------------------|
| Small / medium population, complete list available     | **Simple Random**           |
| Ordered list, want even spread                         | **Systematic**              |
| Important subgroups must be represented                | **Stratified**              |
| Population spread over large area, high travel cost    | **Cluster** or **Multi-stage** |
| Very large hierarchical population (country survey)    | **Multi-stage**             |
| Data arrives as a stream / unknown size                | **Reservoir Sampling**      |
| Units have different importance / size                 | **Weighted / PPS Sampling** |

---

### Final Practical Tips

1. Always prefer the modern `np.random.default_rng()` over the old `np.random.*` functions.
2. Set a `seed` (or `random_state`) when you need reproducibility.
3. In real surveys, **stratified multi-stage designs** are extremely common.
4. After sampling, remember that the sampling design affects how you calculate standard errors and confidence intervals.
